# Daily future-high returns for five cryptocurrencies

This notebook implements the calculation requested by the research supervisor for Bitcoin and four reproducibly sampled cryptocurrencies. For every base UTC date $t$, all four values use the same opening price:

$$D_h(t) = \frac{High_{t+h} - Open_t}{Open_t}, \qquad h \in \{0,1,2,3\}.$$

The values are potential future-high returns, not realized close-to-close returns. They deliberately use future information and therefore belong to outcome/target analysis, never to predictors available at date $t$.

## Interpretation and provisional assumptions

The audio consistently asks for the maximum price as the endpoint, including the formula and final recap. One intermediate reference to the closing price is therefore treated as a verbal slip. No closing prices are used here.

Two reproducibility parameters remain provisional rather than approved methodology: candidates must belong to the approved 40-pair daily collection and have an observation on `2019-07-01`, and the random seed is `20261002`. Bitcoin is always retained. These parameters are explicit in the JSON configuration so they can be changed after confirmation.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display


def find_repository_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'AGENTS.md').exists() and (candidate / 'pyproject.toml').exists():
            return candidate
    raise FileNotFoundError('Could not locate the repository root.')


REPOSITORY_ROOT = find_repository_root()
SOURCE_DIRECTORY = REPOSITORY_ROOT / 'src'
if str(SOURCE_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIRECTORY))

from tcc_crypto.daily_future_high_returns import (
    build_selected_symbol_returns,
    find_symbol_csv,
    load_collection_symbols,
    load_future_high_return_config,
    select_reproducible_symbols,
    symbols_observed_on_date,
)

## Load configuration and select the five pairs

Sampling is deterministic because the configuration records the seed. Re-running the notebook with the same inputs therefore selects the same four pairs.

In [2]:
CONFIG_PATH = REPOSITORY_ROOT / 'configs/binance_daily_future_high_returns.json'
config = load_future_high_return_config(CONFIG_PATH)
collection_config_path = REPOSITORY_ROOT / config.collection_config
raw_directory = REPOSITORY_ROOT / 'data/raw/binance/spot/1d'
collection_symbols = load_collection_symbols(collection_config_path)
eligible_symbols = symbols_observed_on_date(
    collection_symbols, raw_directory, config.analysis_start_utc
)
selected_symbols = select_reproducible_symbols(
    eligible_symbols,
    required_symbol=config.required_symbol,
    additional_symbol_count=config.additional_symbol_count,
    random_seed=config.random_seed,
)

display(pd.DataFrame({
    'Field': ['Analysis ID', 'Period (UTC)', 'Eligible pairs', 'Random seed', 'Selected pairs'],
    'Value': [
        config.analysis_id,
        f'{config.analysis_start_utc.date()} through {config.analysis_end_utc.date()}',
        len(eligible_symbols),
        config.random_seed,
        ', '.join(selected_symbols),
    ],
}))

,Field,Value
0,Analysis ID,binance_daily_future_high_returns_2019H2_2026-...
1,Period (UTC),2019-07-01 through 2026-09-01
2,Eligible pairs,14
3,Random seed,20261002
4,Selected pairs,"BTCUSDT, BNBUSDT, VETUSDT, ATOMUSDT, ZECUSDT"


## Calculate D0, D1, D2, and D3

For each row, `base_open` is the opening price on `date_utc`. `D0` compares it with that date's high; `D1`, `D2`, and `D3` compare the same opening price with the highs one, two, and three calendar days later. Exact UTC dates are used, so a missing candle produces a missing result rather than shifting another date into its place.

`D0` cannot be negative for a valid OHLC candle because its high cannot be below its open. Later horizons can be negative when a future day's high remains below the base opening price.

In [3]:
results = build_selected_symbol_returns(
    selected_symbols,
    raw_directory=raw_directory,
    start_utc=config.analysis_start_utc,
    end_utc=config.analysis_end_utc,
    horizons_days=config.horizons_days,
)
display(results.head(12).style.format({
    'base_open': '{:,.8f}',
    'D0': '{:.4%}',
    'D1': '{:.4%}',
    'D2': '{:.4%}',
    'D3': '{:.4%}',
}))

,symbol,date_utc,base_open,D0,D1,D2,D3
0,BTCUSDT,2019-07-01 00:00:00+00:00,"10,854.10000000",3.9449%,0.7799%,10.4826%,10.5573%
1,BTCUSDT,2019-07-02 00:00:00+00:00,"10,624.90000000",2.9539%,12.8659%,12.9422%,7.3594%
2,BTCUSDT,2019-07-03 00:00:00+00:00,"10,844.98000000",10.5755%,10.6503%,5.1807%,7.5613%
3,BTCUSDT,2019-07-04 00:00:00+00:00,"11,940.00000000",0.5025%,-4.4654%,-2.3032%,-3.3668%
4,BTCUSDT,2019-07-05 00:00:00+00:00,"11,145.67000000",2.3432%,4.6595%,3.5200%,10.6980%
5,BTCUSDT,2019-07-06 00:00:00+00:00,"10,982.41000000",6.2153%,5.0589%,12.3436%,16.5020%
6,BTCUSDT,2019-07-07 00:00:00+00:00,"11,256.45000000",2.5012%,9.6085%,13.6658%,16.7960%
7,BTCUSDT,2019-07-08 00:00:00+00:00,"11,410.00000000",8.1335%,12.1361%,15.2242%,6.1501%
8,BTCUSDT,2019-07-09 00:00:00+00:00,"12,238.60000000",4.5441%,7.4231%,-1.0366%,-2.8892%
9,BTCUSDT,2019-07-10 00:00:00+00:00,"12,548.51000000",4.7700%,-3.4807%,-5.2876%,-5.9651%


## Validate the temporal construction

The checks below verify one row directly against the raw high prices and confirm that the final three base dates retain incomplete future horizons as missing. This prevents accidental wraparound or use of the wrong opening price.

In [4]:
btc = results.loc[results['symbol'] == config.required_symbol].set_index('date_utc')
example_date = config.analysis_start_utc
example = btc.loc[example_date]
assert example['D0'] >= 0
assert results.groupby('symbol').size().nunique() == 1
assert btc.tail(1)[['D1', 'D2', 'D3']].isna().all(axis=None)

display(pd.DataFrame([example], index=[example_date]).style.format({
    'base_open': '{:,.8f}',
    'D0': '{:.4%}',
    'D1': '{:.4%}',
    'D2': '{:.4%}',
    'D3': '{:.4%}',
}))

,symbol,base_open,D0,D1,D2,D3
2019-07-01 00:00:00+00:00,BTCUSDT,"10,854.10000000",3.9449%,0.7799%,10.4826%,10.5573%


## Summary and export

The summary is descriptive only. The CSV preserves returns as decimal fractions (`0.10` means 10%) and timestamps in UTC. Files under `data/raw/` are read but never modified.

In [5]:
return_columns = [f'D{horizon}' for horizon in config.horizons_days]
summary = results.groupby('symbol')[return_columns].agg(['count', 'mean', 'median', 'min', 'max'])
display(summary.style.format('{:.4%}', subset=pd.IndexSlice[:, (slice(None), ['mean', 'median', 'min', 'max'])]))

output_directory = REPOSITORY_ROOT / config.output_directory
output_directory.mkdir(parents=True, exist_ok=True)
output_path = output_directory / 'daily_future_high_returns.csv.gz'
results.to_csv(output_path, index=False, compression='gzip', date_format='%Y-%m-%dT%H:%M:%SZ')
print(f'Wrote {len(results):,} rows to {output_path}')

Wrote 13,100 rows to /Users/marcelapsilverio/projetos/tcc_crypto/outputs/binance_daily_future_high_returns_2019H2_2026-09-01/daily_future_high_returns.csv.gz
